# Chapter 03 · Mathematics: Separate objectives and choose checkpoints

[Chapter guide](README.md) · [Problem-to-mathematics guide](../MATHEMATICS.md) · [Course index](../README.md)

**Status: prepared; mentoring review pending.** The programming lessons are already covered. This companion adds the mathematical view without marking it reviewed yet.

Read one section at a time: words → a small example → symbols → code → your explanation. Symbols are shorthand for ideas you already used. You do not need to memorize all the formulas before continuing.

**Prerequisites:** Mean squared error; “minimum” means the smallest value.

The runnable examples use only Python and the installed PyTorch. Start in a fresh kernel and run Setup, then proceed in order. Each example keeps its calculations small and supplies its own data. Code cells are saved unexecuted so you can make your own predictions first.

## Contents

1. [1. The same loss formula on different data](#split-losses)
2. [2. The supplied targets define the objective](#supplied-targets)
3. [3. Select a checkpoint and interpret patience](#checkpoint)

## Setup

In [ ]:
import torch

<a id="split-losses"></a>
## 1. The same loss formula on different data


Keep $\theta$ for the model's parameter values. $f_\theta(x)$ means its prediction function using those values. The training and validation sets contain different examples, so they give different averages:

$$L_{\mathrm{train}}(\theta)=\frac{1}{n_{\mathrm{train}}}\sum_{i\in\mathrm{train}}(f_\theta(x_i)-y_i)^2$$

$$L_{\mathrm{val}}(\theta)=\frac{1}{n_{\mathrm{val}}}\sum_{i\in\mathrm{val}}(f_\theta(x_i)-y_i)^2$$

The subscripts name which data we use and how many examples it contains. Our optimizer updates parameters using training loss. Validation loss helps us compare checkpoints and settings. We reserve test examples for evaluating the selected approach without using them to choose it.

Validation loss estimates performance on those held-out examples; it is not proof of accuracy on every unseen input. A single larger validation loss does not establish overfitting. Look at the trend across training and consider which examples each set represents.

**Connection:** programming Lessons 01–02 and 05. **Before running:** can the same fixed parameters have different losses on two different sets?


**My prediction or explanation:** _write here_

In [ ]:
# Keep one model fixed while changing which examples we evaluate.
weight, bias = 2.795, 0.92
train_x = torch.tensor([0., 1., 2., 3.])
train_y = torch.tensor([2., 5., 8., 11.])
val_x = torch.tensor([4., 5.])
val_y = torch.tensor([14., 17.])
train_loss = ((weight * train_x + bias - train_y) ** 2).mean()
val_loss = ((weight * val_x + bias - val_y) ** 2).mean()
print("Training loss:", train_loss.item())
print("Validation loss:", val_loss.item())

<a id="supplied-targets"></a>
## 2. The supplied targets define the objective


The optimizer compares predictions with the target values we supply. It does not separately know the rule we intended.

For input $x=3$, weight `1`, and bias `0`, prediction is `3`. A target of `11` gives error `-8`; changing that target to `14` gives error `-11`. The squared loss and its gradients change even though the model and input did not.

A misleading target can change which parameter values minimize training loss. Validation against independent, appropriate targets can reveal problems that training loss alone does not show. The changed-target exercise is one illustration, not a proof that every validation fluctuation is caused by bad labels.

**Connection:** programming Lesson 03. **Before running:** which supplied target asks for a larger upward change from prediction `3`?


**My prediction or explanation:** _write here_

In [ ]:
# Compare the single-example objective before any update.
input_value, prediction = 3.0, 3.0
for target in [11.0, 14.0]:
    error = prediction - target
    print("Target:", target, "| squared loss:", error ** 2,
          "| weight gradient:", 2 * error * input_value,
          "| bias gradient:", 2 * error)

<a id="checkpoint"></a>
## 3. Select a checkpoint and interpret patience


Let $\theta_t$ mean parameter values saved at epoch $t$. Among the checkpoints we evaluated, select the epoch with the smallest validation loss:

$$t^*=\underset{t}{\operatorname{argmin}}\;L_{\mathrm{val}}(\theta_t)$$

“Argmin” means **the index where the minimum occurs**. $t^*$ is the selected epoch, not the numerical loss. It is the best among the checkpoints considered, not a promise about all possible future epochs.

With patience `2`, our simple early-stopping rule stops after two consecutive evaluated epochs fail to improve the best validation value. An improvement resets that counter. This is a practical rule for ending a run; it cannot know whether a later epoch would have improved again.

The code below uses an illustrative sequence, not the measured outputs of our original training notebook. It shows a stop at epoch 4 even though epoch 5 would have been better if training continued.

**Connection:** programming Lessons 04–05 and Chapter 04's checkpoint-selection lesson. **Before running:** which epoch should patience two stop at, and which saved checkpoint should be restored at that point?


**My prediction or explanation:** _write here_

In [ ]:
# Illustrative validation values; lower is better in this exercise.
validation_losses = [0.4, 0.1, 0.2, 0.3, 0.05]
patience = 2
best_loss, best_epoch, no_improvement = float("inf"), None, 0
for epoch, value in enumerate(validation_losses, start=1):
    if value < best_loss:
        best_loss, best_epoch = value, epoch
        no_improvement = 0
    else:
        no_improvement += 1
    print("Epoch:", epoch, "| loss:", value, "| no-improvement count:", no_improvement)
    if no_improvement >= patience:
        stopped_epoch = epoch
        break
print("Stopped epoch:", stopped_epoch)
print("Checkpoint to restore:", best_epoch)
print("Best epoch if all listed epochs were evaluated:",
      min(range(1, len(validation_losses) + 1), key=lambda epoch: validation_losses[epoch - 1]))

**My observation:** _write here_

Explain why choosing the last training step is different from choosing the lowest validation loss, and why a test set must not become another set used repeatedly to choose parameters or settings.